# Multi-task Grad-CAM for earthworm classification

This notebook loads a trained multi-task model and plots Grad-CAM maps for the available tasks, for example `genus`, `species`, and `age`.

Run it from the project root, where `src/` is available.

In [ ]:
from __future__ import annotations

import copy
import math
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT_CANDIDATES = [
    parent.resolve()
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "config.yaml").is_file()
    and (parent / "src" / "worm_species").is_dir()
]
if not PROJECT_ROOT_CANDIDATES:
    raise FileNotFoundError(
        "Could not locate repository root containing config.yaml and src/worm_species."
    )
PROJECT_ROOT = PROJECT_ROOT_CANDIDATES[0]
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from worm_species.data.labels import get_target_cols
from worm_species.data.metadata import prepare_metadata
from worm_species.data.transforms import build_transforms
from worm_species.models import build_model

In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

RUN_DIR = PROJECT_ROOT / Path(
    "outputs_slurm/node_local_sweep_20260706_122839/"
    "run_117/vit_b_16__rel_path_seg__genus__755154d1"
)

CKPT_PATH = RUN_DIR / "best_model.pt"
CONFIG_PATH = RUN_DIR / "config.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The notebook will plot only tasks that actually exist in the checkpoint.
TASKS_TO_PLOT = ["genus", "species", "age"]

# "pred" = Grad-CAM for the predicted class
# "true" = Grad-CAM for the true class, when available
CAM_MODE = "pred"

N_RANDOM_EXAMPLES = 8
RANDOM_STATE = 42

OUT_DIR = PROJECT_ROOT / "figures" / "gradcam_multitask_all_tasks" / RUN_DIR.name
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Output directory:", OUT_DIR)

## Model definition

This reproduces the multi-task wrapper used during training: one shared image backbone and one classification head per task.

In [ ]:
class MultiTaskClassifier(nn.Module):
    """
    Shared image backbone with one classification head per task.

    This should match the structure used in your multi-task training script.
    """

    def __init__(self, base_model: nn.Module, num_classes_by_task: dict[str, int]):
        super().__init__()
        self.backbone = base_model
        feature_dim = self._remove_classifier_and_get_feature_dim()

        self.heads = nn.ModuleDict({
            task: nn.Linear(feature_dim, num_classes)
            for task, num_classes in num_classes_by_task.items()
        })

    def _remove_classifier_and_get_feature_dim(self) -> int:
        # ResNet-style models
        if hasattr(self.backbone, "fc") and isinstance(self.backbone.fc, nn.Linear):
            feature_dim = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
            return feature_dim

        # EfficientNet, ConvNeXt, MobileNet, DenseNet-style models
        if hasattr(self.backbone, "classifier"):
            classifier = self.backbone.classifier

            if isinstance(classifier, nn.Linear):
                feature_dim = classifier.in_features
                self.backbone.classifier = nn.Identity()
                return feature_dim

            if isinstance(classifier, nn.Sequential):
                for i in range(len(classifier) - 1, -1, -1):
                    if isinstance(classifier[i], nn.Linear):
                        feature_dim = classifier[i].in_features
                        classifier[i] = nn.Identity()
                        return feature_dim

        # torchvision ViT-style models
        if hasattr(self.backbone, "heads") and hasattr(self.backbone.heads, "head"):
            head = self.backbone.heads.head
            if isinstance(head, nn.Linear):
                feature_dim = head.in_features
                self.backbone.heads.head = nn.Identity()
                return feature_dim

        raise ValueError(
            "Could not identify the final classifier layer. "
            "Add a case in MultiTaskClassifier._remove_classifier_and_get_feature_dim "
            "for this model."
        )

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        features = self.backbone(x)

        if isinstance(features, (tuple, list)):
            features = features[0]

        if features.ndim > 2:
            features = torch.flatten(
                torch.nn.functional.adaptive_avg_pool2d(features, 1),
                1,
            )

        return {
            task: head(features)
            for task, head in self.heads.items()
        }


def build_multitask_model(
    cfg: dict,
    num_classes_by_task: dict[str, int],
) -> nn.Module:
    """
    Build the same multi-task model used during training.
    """

    temporary_num_classes = max(num_classes_by_task.values())

    base_model = build_model(
        name=cfg["model"]["name"],
        num_classes=temporary_num_classes,
        pretrained=cfg["model"].get("pretrained", True),
        freeze_backbone=cfg["model"].get("freeze_backbone", False),
    )

    return MultiTaskClassifier(
        base_model=base_model,
        num_classes_by_task=num_classes_by_task,
    )

## Load checkpoint, config, and labels

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

if "cfg" in ckpt:
    cfg = ckpt["cfg"]
else:
    with open(CONFIG_PATH, "r") as f:
        cfg = json.load(f)

if "label_to_index_by_task" in ckpt:
    label_to_index_by_task = ckpt["label_to_index_by_task"]
else:
    with open(RUN_DIR / "label_to_index_by_task.json", "r") as f:
        label_to_index_by_task = json.load(f)

available_tasks = list(label_to_index_by_task.keys())

# Build the model with all heads stored in the checkpoint.
num_classes_by_task_all = {
    task: len(label_to_index)
    for task, label_to_index in label_to_index_by_task.items()
}

# Plot only requested tasks that exist.
TASKS = [task for task in TASKS_TO_PLOT if task in available_tasks]

if len(TASKS) == 0:
    raise ValueError(
        f"None of TASKS_TO_PLOT={TASKS_TO_PLOT} were found. "
        f"Available tasks are: {available_tasks}"
    )

index_to_label_by_task = {
    task: {int(v): k for k, v in label_to_index_by_task[task].items()}
    for task in available_tasks
}

print("Available tasks:", available_tasks)
print("Tasks to plot:", TASKS)

for task in TASKS:
    print(f"{task}: {len(label_to_index_by_task[task])} classes")

In [ ]:
# Avoid downloading pretrained weights when reconstructing the model for inference.
cfg_for_model = copy.deepcopy(cfg)
cfg_for_model.setdefault("model", {})
cfg_for_model["model"]["pretrained"] = False

model = build_multitask_model(
    cfg=cfg_for_model,
    num_classes_by_task=num_classes_by_task_all,
)

state_dict = ckpt["model_state"] if "model_state" in ckpt else ckpt
state_dict = {
    k.replace("module.", ""): v
    for k, v in state_dict.items()
}

model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)
model.eval()

print("Loaded model successfully.")
print("Model name:", cfg_for_model["model"]["name"])

if "selection_metric" in ckpt:
    print("Selection metric:", ckpt["selection_metric"])
if "best_val_score" in ckpt:
    print("Best validation score:", ckpt["best_val_score"])
if "best_epoch" in ckpt:
    print("Best epoch:", ckpt["best_epoch"])

In [ ]:
cfg

## Load metadata and image transform

In [ ]:
cfg['data']['root_dir'] = '/mnt/extssd/Earthworms/petridish-worm-images'
cfg['data']['metadata_csv'] = '/mnt/extssd/Earthworms/petridish-worm-images/01_Segmented/global_metadata.csv'
df = prepare_metadata(cfg)

target_cols = get_target_cols(cfg)
root_dir = Path(cfg["data"]["root_dir"])
image_col = cfg["data"]["image_col"]
image_size = cfg["data"]["image_size"]

eval_tf = build_transforms(image_size=image_size, train=False)

print("Target columns:")
for task in TASKS:
    print(f"  {task}: {target_cols[task]}")

print("Image column:", image_col)
print("Number of metadata rows:", len(df))

In [ ]:
def resolve_image_path(row: pd.Series) -> Path:
    p = Path(row[image_col])
    if p.is_absolute():
        return p
    return root_dir / p


def load_row_image(row: pd.Series) -> tuple[Image.Image, torch.Tensor, Path]:
    image_path = resolve_image_path(row)
    image = Image.open(image_path).convert("RGB")

    x = eval_tf(image)

    # In case your transform returns a dict-like object.
    if isinstance(x, dict):
        x = x["image"]

    x = x.unsqueeze(0).to(DEVICE)

    return image, x, image_path


def get_true_label(row: pd.Series, task: str) -> str | None:
    col = target_cols[task]

    if col not in row.index:
        return None

    value = row[col]

    if pd.isna(value):
        return None

    return str(value)


def get_true_index(row: pd.Series, task: str) -> int | None:
    true_label = get_true_label(row, task)

    if true_label is None:
        return None

    label_to_index = label_to_index_by_task[task]

    if true_label not in label_to_index:
        return None

    return int(label_to_index[true_label])


def row_has_label_for_any_task(row: pd.Series) -> bool:
    return any(get_true_label(row, task) is not None for task in TASKS)


def row_has_label_for_all_tasks(row: pd.Series) -> bool:
    return all(get_true_label(row, task) is not None for task in TASKS)


labelled_df = df[df.apply(row_has_label_for_any_task, axis=1)].reset_index(drop=True)

if len(labelled_df) == 0:
    raise ValueError("No rows have labels for the selected tasks.")

print("Rows with at least one selected-task label:", len(labelled_df))

## Prediction helpers

In [ ]:
def get_task_logits(model_output: dict[str, torch.Tensor], task: str) -> torch.Tensor:
    if not isinstance(model_output, dict):
        raise TypeError(
            "Expected the model to return a dictionary of task logits, "
            "e.g. {'genus': logits, 'species': logits, 'age': logits}."
        )

    if task not in model_output:
        raise KeyError(
            f"Task '{task}' not found in model output. "
            f"Available outputs: {list(model_output.keys())}"
        )

    return model_output[task]


def predict_task(x: torch.Tensor, task: str):
    model.eval()

    with torch.no_grad():
        output = model(x)
        logits = get_task_logits(output, task)
        probs = torch.softmax(logits, dim=1).squeeze().detach().cpu().numpy()

    pred_idx = int(np.argmax(probs))
    pred_label = index_to_label_by_task[task][pred_idx]
    pred_prob = float(probs[pred_idx])

    return pred_idx, pred_label, pred_prob, probs


def print_top_probabilities_for_row(row: pd.Series):
    image, x, image_path = load_row_image(row)

    print("Image:", image_path)

    for task in TASKS:
        pred_idx, pred_label, pred_prob, probs = predict_task(x, task)

        print(f"\nTask: {task}")
        print("True:", get_true_label(row, task))
        print("Pred:", pred_label)
        print(f"Pred probability: {pred_prob:.4f}")

        top_probs = sorted(
            [
                (index_to_label_by_task[task][i], float(p))
                for i, p in enumerate(probs)
            ],
            key=lambda z: z[1],
            reverse=True,
        )

        print("Top probabilities:")
        for label, p in top_probs:
            print(f"  {label}: {p:.4f}")

## Grad-CAM utilities

The target layer is selected automatically by looking for the last layer with a spatial feature map.

In [ ]:
def _tokens_to_spatial(t: torch.Tensor) -> torch.Tensor:
    """
    Convert ViT-like token output [B, N, C] to [B, C, H, W].
    Removes CLS token if N - 1 is a square.
    """

    if t.ndim != 3:
        raise ValueError(f"Expected [B, N, C], got shape {tuple(t.shape)}")

    b, n, c = t.shape

    # Try removing CLS token.
    h = int(math.sqrt(n - 1))
    if h * h == n - 1:
        t = t[:, 1:, :]
        return t.transpose(1, 2).reshape(b, c, h, h)

    # Try using all tokens.
    h = int(math.sqrt(n))
    if h * h == n:
        return t.transpose(1, 2).reshape(b, c, h, h)

    raise ValueError(
        f"Cannot reshape token output with N={n} into a square spatial map."
    )


def activation_to_spatial(t: torch.Tensor) -> torch.Tensor:
    """
    Converts either:
    - CNN activation [B, C, H, W]
    - ViT-like tokens [B, N, C]

    into [B, C, H, W].
    """

    if t.ndim == 4:
        return t

    if t.ndim == 3:
        return _tokens_to_spatial(t)

    raise ValueError(f"Unsupported activation shape: {tuple(t.shape)}")


def find_gradcam_target_layer(model: nn.Module, example_x: torch.Tensor):
    """
    Finds the last layer with a spatial output.

    Preference:
    1. last 4D CNN-like feature map
    2. last 3D token output that can be reshaped to a square map
    """

    model.eval()

    candidates_4d = []
    candidates_3d = []
    hooks = []

    def make_hook(name, module):
        def hook(module, inputs, output):
            if not isinstance(output, torch.Tensor):
                return

            if output.ndim == 4:
                candidates_4d.append((name, module, tuple(output.shape)))

            elif output.ndim == 3:
                try:
                    _ = activation_to_spatial(output.detach())
                    candidates_3d.append((name, module, tuple(output.shape)))
                except Exception:
                    pass

        return hook

    for name, module in model.named_modules():
        if name == "":
            continue
        hooks.append(module.register_forward_hook(make_hook(name, module)))

    with torch.no_grad():
        _ = model(example_x)

    for h in hooks:
        h.remove()

    if len(candidates_4d) > 0:
        name, layer, shape = candidates_4d[-1]
    elif len(candidates_3d) > 0:
        name, layer, shape = candidates_3d[-1]
    else:
        raise ValueError(
            "Could not find a suitable Grad-CAM target layer. "
            "No 4D feature maps or reshapeable 3D token maps were found."
        )

    print("Grad-CAM target layer:")
    print(" ", name, shape)

    return layer

In [ ]:
class MultiTaskGradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = target_layer.register_forward_hook(self._forward_hook)
        self.backward_handle = target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def __call__(
        self,
        x: torch.Tensor,
        task: str,
        class_idx: int | None = None,
    ):
        self.model.zero_grad(set_to_none=True)

        output = self.model(x)
        logits = get_task_logits(output, task)

        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())

        score = logits[:, class_idx]
        score.backward()

        activations = activation_to_spatial(self.activations)
        gradients = activation_to_spatial(self.gradients)

        weights = gradients.mean(dim=(2, 3), keepdim=True)

        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = F.interpolate(
            cam,
            size=x.shape[2:],
            mode="bilinear",
            align_corners=False,
        )

        cam = cam.squeeze().detach().cpu().numpy()

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        probs = torch.softmax(logits.detach(), dim=1).squeeze().cpu().numpy()

        return cam, class_idx, probs


def get_cam_for_task(
    x: torch.Tensor,
    task: str,
    class_idx: int,
    target_layer: nn.Module,
):
    gradcam = MultiTaskGradCAM(model, target_layer)
    cam_arr, _, probs = gradcam(x, task=task, class_idx=class_idx)
    gradcam.remove_hooks()
    return cam_arr, probs

## Visualisation helpers

In [ ]:
def overlay_cam_on_image(
    pil_image: Image.Image,
    cam_arr: np.ndarray,
    alpha: float = 0.45,
) -> np.ndarray:
    image = pil_image.resize((cam_arr.shape[1], cam_arr.shape[0]))
    image_np = np.asarray(image).astype(np.float32) / 255.0

    heatmap = cm.get_cmap("jet")(cam_arr)[..., :3]
    overlay = (1.0 - alpha) * image_np + alpha * heatmap
    overlay = np.clip(overlay, 0.0, 1.0)

    return overlay


def class_idx_for_cam(row: pd.Series, task: str, pred_idx: int, pred_label: str):
    true_label = get_true_label(row, task)
    true_idx = get_true_index(row, task)

    if CAM_MODE == "true":
        if true_idx is None:
            return pred_idx, f"pred CAM: {pred_label}"
        return true_idx, f"true CAM: {true_label}"

    if CAM_MODE == "pred":
        return pred_idx, f"pred CAM: {pred_label}"

    raise ValueError("CAM_MODE must be either 'pred' or 'true'.")

In [ ]:
def plot_all_tasks_for_row(
    row: pd.Series,
    target_layer: nn.Module,
    save_path: Path | None = None,
):
    image, x, image_path = load_row_image(row)

    n_cols = len(TASKS) + 1
    fig = plt.figure(figsize=(5 * n_cols, 5))

    # Original image
    ax = plt.subplot(1, n_cols, 1)
    ax.imshow(image)
    ax.axis("off")

    true_text = "\n".join([
        f"{task}: {get_true_label(row, task)}"
        for task in TASKS
    ])

    ax.set_title(
        f"Original\n{image_path.name}\n{true_text}",
        fontsize=8,
    )

    # CAM per task
    for plot_i, task in enumerate(TASKS, start=2):
        pred_idx, pred_label, pred_prob, probs = predict_task(x, task)

        true_label = get_true_label(row, task)

        idx_for_cam, cam_label = class_idx_for_cam(
            row=row,
            task=task,
            pred_idx=pred_idx,
            pred_label=pred_label,
        )

        cam_arr, _ = get_cam_for_task(
            x=x,
            task=task,
            class_idx=idx_for_cam,
            target_layer=target_layer,
        )

        overlay = overlay_cam_on_image(image, cam_arr)

        if true_label is None:
            status = "no true label"
        else:
            status = "correct" if str(true_label) == str(pred_label) else "wrong"

        ax = plt.subplot(1, n_cols, plot_i)
        ax.imshow(overlay)
        ax.axis("off")
        ax.set_title(
            f"{task}\n"
            f"True: {true_label}\n"
            f"Pred: {pred_label}\n"
            f"P = {pred_prob:.3f} | {status}\n"
            f"{cam_label}",
            fontsize=9,
        )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()
    plt.close(fig)

In [ ]:
def plot_random_examples_all_tasks(
    df_in: pd.DataFrame,
    target_layer: nn.Module,
    n_examples: int = 8,
    random_state: int = 42,
    save_path: Path | None = None,
):
    sampled_rows = df_in.sample(
        n=min(n_examples, len(df_in)),
        random_state=random_state,
    ).reset_index(drop=True)

    n_rows = len(sampled_rows)
    n_cols = len(TASKS) + 1

    fig = plt.figure(figsize=(5 * n_cols, 4.8 * n_rows))

    for row_i, (_, row) in enumerate(sampled_rows.iterrows()):
        image, x, image_path = load_row_image(row)

        # Original image
        plot_i = row_i * n_cols + 1
        ax = plt.subplot(n_rows, n_cols, plot_i)
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(
            f"Original\n{image_path.name}",
            fontsize=8,
        )

        # One CAM per task
        for task_j, task in enumerate(TASKS, start=2):
            pred_idx, pred_label, pred_prob, probs = predict_task(x, task)

            true_label = get_true_label(row, task)

            idx_for_cam, cam_label = class_idx_for_cam(
                row=row,
                task=task,
                pred_idx=pred_idx,
                pred_label=pred_label,
            )

            cam_arr, _ = get_cam_for_task(
                x=x,
                task=task,
                class_idx=idx_for_cam,
                target_layer=target_layer,
            )

            overlay = overlay_cam_on_image(image, cam_arr)

            if true_label is None:
                status = "no true label"
            else:
                status = "correct" if str(true_label) == str(pred_label) else "wrong"

            plot_i = row_i * n_cols + task_j
            ax = plt.subplot(n_rows, n_cols, plot_i)
            ax.imshow(overlay)
            ax.axis("off")
            ax.set_title(
                f"{task} | {cam_label}\n"
                f"True: {true_label}\n"
                f"Pred: {pred_label}\n"
                f"P = {pred_prob:.2f} | {status}",
                fontsize=8,
            )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()
    plt.close(fig)

## Find Grad-CAM target layer

In [ ]:
example_row = labelled_df.sample(1, random_state=RANDOM_STATE).iloc[0]
example_image, example_x, example_image_path = load_row_image(example_row)

target_layer = find_gradcam_target_layer(model, example_x)

## One image: all tasks

In [ ]:
print_top_probabilities_for_row(example_row)

plot_all_tasks_for_row(
    row=example_row,
    target_layer=target_layer,
    save_path=OUT_DIR / f"one_example_all_tasks_{CAM_MODE}_cam.png",
)

## Random examples: all tasks

In [ ]:
plot_random_examples_all_tasks(
    df_in=labelled_df,
    target_layer=target_layer,
    n_examples=N_RANDOM_EXAMPLES,
    random_state=RANDOM_STATE,
    save_path=OUT_DIR / f"random_examples_all_tasks_{CAM_MODE}_cam.png",
)

## Optional: plot one manually chosen metadata row

Change `ROW_INDEX` to inspect a specific row in the metadata table.

In [ ]:
ROW_INDEX = 0

manual_row = labelled_df.iloc[ROW_INDEX]

print_top_probabilities_for_row(manual_row)

plot_all_tasks_for_row(
    row=manual_row,
    target_layer=target_layer,
    save_path=OUT_DIR / f"manual_row_{ROW_INDEX}_all_tasks_{CAM_MODE}_cam.png",
)